# UdaPlay — Part 1: Offline RAG over the Video Game Catalogue

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# Environment variables required for this notebook (place them in `.env`):
#   OPENAI_API_KEY="sk-..."          # used by the OpenAI embedding function below
#   CHROMA_OPENAI_API_KEY="sk-..."   # convenience key Chroma's helper auto-reads
#   TAVILY_API_KEY="tvly-..."        # only used in Part 2; harmless here
#
# Copy `.env.example` to `.env` at the project root and fill in the values
# before running this notebook.


In [4]:
# Load environment variables and verify the OpenAI key is present, since the
# Chroma OpenAIEmbeddingFunction below needs it to compute embeddings.
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Copy .env.example to .env and add your key."
    )

# Chroma's helper reads CHROMA_OPENAI_API_KEY when present, otherwise falls
# back to OPENAI_API_KEY. Mirror the value so the helper Just Works.
os.environ.setdefault("CHROMA_OPENAI_API_KEY", os.environ["OPENAI_API_KEY"])


### VectorDB Instance

In [5]:
# A persistent client writes its collections to disk under ./chromadb,
# so re-running Part 2 can pick the same vector store back up without
# having to re-embed every JSON file.
chroma_client = chromadb.PersistentClient(path="chromadb")
print(f"ChromaDB persistent client ready at: {os.path.abspath('chromadb')}")


ChromaDB persistent client ready at: /home/student/udaplay/chromadb


### Collection

In [6]:
# Use OpenAI's `text-embedding-3-small` for embeddings. It's a good
# balance between quality and cost for short game descriptions.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-small",
)


In [7]:
# `get_or_create_collection` lets us re-run the notebook without raising
# "Collection already exists". The same name "udaplay" is used in Part 2.
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
    metadata={"description": "UdaPlay video game catalogue (Part 1 RAG)."},
)
print(f"Collection '{collection.name}' ready (current document count: {collection.count()}).")


Collection 'udaplay' ready (current document count: 0).


### Add documents

In [8]:
# Ingest every game JSON into the collection.
#
# The loop is idempotent: existing IDs are skipped, so re-running the
# notebook is safe and cheap (no duplicate embeddings, no wasted tokens).

data_dir = "games"
existing_ids = set(collection.get()["ids"])
added, skipped = 0, 0

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    doc_id = os.path.splitext(file_name)[0]
    if doc_id in existing_ids:
        skipped += 1
        continue

    # Index a single content string per game; the metadata dict keeps the
    # structured fields available for later filtering / pretty-printing.
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) "
        f"- {game['Description']}"
    )

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game],
    )
    added += 1

print(f"Ingest complete. Added: {added}, skipped (already present): {skipped}.")
print(f"Collection now contains {collection.count()} documents.")


Ingest complete. Added: 20, skipped (already present): 0.
Collection now contains 20 documents.


### Demonstrate semantic search

Query the collection with three natural-language questions and pretty-print the top hits + cosine distances.

In [9]:
def pretty_print_hits(question, results, top_k=3):
    print(f"\n>>> {question}")
    docs = results["documents"][0][:top_k]
    metas = results["metadatas"][0][:top_k]
    dists = results["distances"][0][:top_k]
    for rank, (doc, meta, dist) in enumerate(zip(docs, metas, dists), start=1):
        print(
            f"  {rank}. ({1 - dist:.3f} sim) {meta['Name']} — {meta['Platform']} "
            f"({meta['YearOfRelease']})"
        )
        print(f"     {doc}")


for q in [
    "Which Pokémon games were released on the Game Boy Color?",
    "What is the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]:
    res = collection.query(query_texts=[q], n_results=3)
    pretty_print_hits(q, res)



>>> Which Pokémon games were released on the Game Boy Color?
  1. (0.842 sim) Pokémon Crystal — Game Boy Color (2000)
     [Game Boy Color] Pokémon Crystal (2000) - The third installment in the Pokémon Gold and Silver series, featuring enhanced gameplay and graphics.
  2. (0.811 sim) Pokémon Gold and Silver — Game Boy Color (1999)
     [Game Boy Color] Pokémon Gold and Silver (1999) - The second generation of Pokémon games, introducing new Pokémon and a real-time clock system.
  3. (0.529 sim) Pokémon Red and Blue — Game Boy (1996)
     [Game Boy] Pokémon Red and Blue (1996) - The original Pokémon games that started the franchise, featuring the iconic Generation I creatures.

>>> What is the first 3D platformer Mario game?
  1. (0.806 sim) Super Mario 64 — Nintendo 64 (1996)
     [Nintendo 64] Super Mario 64 (1996) - A revolutionary 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
  2. (0.498 sim) Super Mario World — Super Nintendo 

### Persistence sanity check

Re-open the persistent client to confirm the data survives a fresh client instance — important for Part 2, which only opens the existing collection (it does not re-ingest).

In [10]:
fresh_client = chromadb.PersistentClient(path="chromadb")
fresh_collection = fresh_client.get_collection(name="udaplay", embedding_function=embedding_fn)
print(f"Fresh client sees {fresh_collection.count()} documents in '{fresh_collection.name}'.")


Fresh client sees 20 documents in 'udaplay'.
